# CardioBench on Kaggle — CardiacNet (ASD / PAH)

Runs the CardioBench zero-shot and linear-probe protocols on CardiacNet with a free Kaggle GPU and
scores them with the benchmark's own `evaluation/cardiacnet.py` (1000× bootstrap, 95% CI).

**Before running**
1. *Add Input* → search **AbnormCardiacEchoVideos** (`xiaoweixumedicalai/abnormcardiacechovideos`) → Add.
2. *Settings* → **Accelerator: GPU T4 x2** (or P100) and **Internet: On** (needed for `git clone` and the Hugging Face weights).
3. *Run All*. Embeddings are cached in `/kaggle/working`, so re-running after an interruption resumes.

Per model it produces: `zeroshot` (prompt similarity), `probe` (logistic regression on mean-pooled
frame embeddings) and `probe-meanstd` (mean + std over frames), plus a label-efficiency curve
(5 %→100 % of the training labels, 5 seeds).

Two frame-sampling settings are run: `consecutive` (the first 16 frames, the repo default,
~15 % of a typical clip) and `uniform` (16 frames spread over the whole clip). Every result is
reported on the full test split and on `noPHI`, which drops the `PHI*` control clips — a
separate RGB source that contains only negatives and is therefore a shortcut.

In [ ]:
# ---- configuration -------------------------------------------------------
REPO_URL = "https://github.com/emirrcodes/CardioBench"
BRANCH = "main"
MODELS = ["echo_clip", "biomed_clip", "siglip2"]   # aliases from src/models.py
DATA_ROOT = "/kaggle/input"                          # the script finds CardiacNet/ below this
OUT_DIR = "/kaggle/working/cardiacnet"   # one sub-run per sampling: {OUT_DIR}_{sampling}
MAX_FRAMES = 16
SAMPLINGS = ["consecutive", "uniform"]
MODELS_STR = " ".join(MODELS)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /kaggle/working/CardioBench && git clone -q --depth 1 -b {BRANCH} {REPO_URL} /kaggle/working/CardioBench
# Kaggle already ships torch/transformers; only add what is missing (no torch pin).
!pip install -q "open_clip_torch>=2.31" nibabel
%cd /kaggle/working/CardioBench
!git log --oneline -1

In [ ]:
import subprocess, sys
for sampling in SAMPLINGS:
    cmd = [sys.executable, "scripts/cardiacnet_benchmark.py", "--data_root", DATA_ROOT,
           "--out_dir", f"{OUT_DIR}_{sampling}", "--models", *MODELS,
           "--max_frames", str(MAX_FRAMES), "--sampling", sampling,
           "--device", "cuda", "--precision", "fp16"]
    print("#" * 20, sampling, "#" * 20, flush=True)
    subprocess.run(cmd, check=True)

## Results

In [ ]:
import pandas as pd
from IPython.display import Image, display

summary = pd.concat([pd.read_csv(f"{OUT_DIR}_{s}/summary.csv").assign(sampling=s) for s in SAMPLINGS])
summary.to_csv("/kaggle/working/summary_all_runs.csv", index=False)
table = (summary.pivot_table(index=["task", "model", "protocol"], columns=["test_set", "sampling"],
                             values="AUROC(%)").round(1))
display(table)
for s in SAMPLINGS:
    for fig in ["auroc_by_protocol.png", "auroc_by_protocol_noPHI.png", "label_efficiency.png"]:
        print(s, fig); display(Image(f"{OUT_DIR}_{s}/figures/{fig}"))

In [ ]:
# Label efficiency table: mean ± std test AUROC over seeds.
for s in SAMPLINGS:
    curves = pd.read_csv(f"{OUT_DIR}_{s}/label_efficiency.csv")
    tab = (curves.groupby(["task", "model", "protocol", "fraction"])["auroc"]
           .agg(lambda x: f"{100*x.mean():.1f} ± {100*x.std(ddof=0):.1f}")
           .unstack("fraction"))
    print(s); display(tab)

In [ ]:
# Bundle everything except the (large) embeddings for download from the Output tab.
!cd /kaggle/working && zip -qr cardiacnet_results.zip summary_all_runs.csv cardiacnet_*/summary.csv cardiacnet_*/label_efficiency.csv cardiacnet_*/eval* cardiacnet_*/figures cardiacnet_*/predictions cardiacnet_*/curves
!ls -lh /kaggle/working/cardiacnet_results.zip